# XOR, separated by a product

The four XOR points are not separable by a straight line. Points with the same signs are one class, and points with opposite signs are the other.

| Point | Product $x_1 x_2$ | Label |
|---|---:|---:|
| $(1, 1)$ | $+1$ | $-1$ |
| $(1, -1)$ | $-1$ | $+1$ |
| $(-1, -1)$ | $+1$ | $-1$ |
| $(-1, 1)$ | $-1$ | $+1$ |

Lift each point with

$$
\psi(x_1, x_2) = (x_1,\ x_2,\ x_1 x_2)
$$

In that space the weight $\mathbf{w} = (0, 0, -1)$ and bias $b = 0$ score

$$
f(x_1, x_2) = -x_1 x_2
$$

Every training functional margin equals $1$. The objective is $\tfrac{1}{2}\|\mathbf{w}\|^2 = 1/2$. No shorter weight can keep those margins at $1$, because the product coordinate is already $\pm 1$ and the weight on it is already $\pm 1$.


In [1]:
# Every XOR point has functional margin 1 under f = -x1 x2.
rows = [((1, 1), -1), ((1, -1), 1), ((-1, -1), -1), ((-1, 1), 1)]
for (x1, x2), label in rows:
    score = -(x1 * x2)
    assert label * score == 1
    print((x1, x2), "score", score, "margin", label * score)


(1, 1) score -1 margin 1
(1, -1) score 1 margin 1
(-1, -1) score -1 margin 1
(-1, 1) score 1 margin 1


## Multipliers in the product space

Stationarity says the third coordinate of $\mathbf{w}$ equals $-1$, and the first two equal $0$. With one multiplier $\alpha$ on each point,

$$
\sum_i \alpha y_i (x_1 x_2)_i = -4\alpha = -1
$$

so $\alpha = 1/4$. The labels are balanced, two plus and two minus, so $\sum \alpha_i y_i = 0$. The squared-length identity $\|\mathbf{w}\|^2 = 1$ makes the dual objective

$$
\sum_i \alpha_i - \frac{1}{2}\|\mathbf{w}\|^2 = 1 - \frac{1}{2} = \frac{1}{2}
$$

which matches the primal. The kernel that this map actually uses is

$$
K_\psi(\mathbf{x}, \mathbf{z}) = \mathbf{x}\cdot\mathbf{z} + (x_1 x_2)(z_1 z_2)
$$

not $(\mathbf{x}\cdot\mathbf{z})^2$.


In [2]:
# Recover w from alpha = 1/4 and compare the two kernels at a new point.
from fractions import Fraction

rows = [((1, 1), -1), ((1, -1), 1), ((-1, -1), -1), ((-1, 1), 1)]
alpha = Fraction(1, 4)
w = [Fraction(0), Fraction(0), Fraction(0)]
for (x1, x2), label in rows:
    features = (x1, x2, x1 * x2)
    for j in range(3):
        w[j] += alpha * label * features[j]
assert w == [0, 0, -1]
assert sum(alpha * label for _, label in rows) == 0

def product_kernel(x, z):
    return x[0] * z[0] + x[1] * z[1] + (x[0] * x[1]) * (z[0] * z[1])

new_point = (2, 1)
# f = sum alpha y K(x_i, new) because b = 0.
score = sum(alpha * label * product_kernel(point, new_point) for point, label in rows)
assert score == -2
assert score == -(new_point[0] * new_point[1])
print("recovered w", w)
print("score at (2, 1)", score)


recovered w [Fraction(0, 1), Fraction(0, 1), Fraction(-1, 1)]
score at (2, 1) -2


## The textbook polynomial kernel on the same pattern

If you insist on $K(\mathbf{x}, \mathbf{z}) = (\mathbf{x}\cdot\mathbf{z})^2$, use the stretched map $(x_1^2,\ \sqrt{2} x_1 x_2,\ x_2^2)$. The decision rule on these four points is still $f = -x_1 x_2$, but the cross coordinate is longer by $\sqrt{2}$, so the multiplier on that coordinate is $-1/\sqrt{2}$ and each $\alpha$ drops from $1/4$ to $1/8$. Same signs, different bookkeeping. Do not mix the two alphas.


In [3]:
# (x·z)^2 uses alpha 1/8 and still scores (2, 1) as -2.
points = [(1, 1), (1, -1), (-1, -1), (-1, 1)]
labels = [-1, 1, -1, 1]
alpha = 1 / 8

def squared_dot(x, z):
    return (x[0] * z[0] + x[1] * z[1]) ** 2

score = sum(
    alpha * y * squared_dot(point, (2, 1))
    for point, y in zip(points, labels)
)
assert abs(score - (-2)) < 1e-12
print("polynomial-kernel score", score)


polynomial-kernel score -2.0


## A pitfall

A kernel matrix built from the wrong formula can look smooth and still implement a different feature map. When the arithmetic is small, expand one row by hand before trusting a library call.

## What to carry forward

XOR is linear in the product coordinate: $f = -x_1 x_2$, with $\alpha = 1/4$ for $K_\psi$ and $\alpha = 1/8$ for $(\mathbf{x}\cdot\mathbf{z})^2$. Both score $(2, 1)$ as $-2$.
